# Notebook 01: Real Automated Metrics vs. Real Human-Style Correctness Labels

`[REAL]` Companion to Modules 01-02. Real `gpt-4o-mini` calls generating real, varied-phrasing answers to a real, small set of factual questions, scored under real BLEU-1/ROUGE-1 and a real, rigorous, pre-stated correctness protocol -- testing whether Module 02's small hand-worked counterexample (n-gram overlap can favor a wrong answer over a correct rephrasing) generalizes across a larger real sample.

**Correctness labeling protocol (stated before any generation, per the signed-off plan):** each question has one authoritative real reference fact, fixed in advance. A real generated answer is labeled correct **only if** the authoritative fact string (case-insensitive) appears literally in the generated text -- an explicit, reproducible, defensible real rule, not a subjective judgment applied after seeing results.

**Sample-size caveat, stated upfront:** this notebook's real sample (8 questions x 2 real generations each = 16 items) is small. Any correlation finding below is reported as **exploratory**, not a statistically robust conclusion -- Module 02's own small, hand-verified counterexample remains the real, load-bearing evidence regardless of how this larger sample's trend comes out.

In [1]:
import os
import math
from collections import Counter
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv())
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-4o-mini"
print(f"OpenAI client ready. Model: {MODEL}")

OpenAI client ready. Model: gpt-4o-mini


## 1. Real Question Set with Pre-Fixed Authoritative Facts

`[REAL]` 8 real factual questions, each with one authoritative real fact string and a real reference sentence (for BLEU/ROUGE scoring) -- both fixed **before** any real generation happens.

In [2]:
QUESTIONS = [
    {"q": "What is the capital of France?", "fact": "Paris", "reference": "The capital of France is Paris."},
    {"q": "What is the chemical symbol for gold?", "fact": "Au", "reference": "The chemical symbol for gold is Au."},
    {"q": "Who wrote Romeo and Juliet?", "fact": "Shakespeare", "reference": "Romeo and Juliet was written by William Shakespeare."},
    {"q": "What is the largest planet in our solar system?", "fact": "Jupiter", "reference": "The largest planet in our solar system is Jupiter."},
    {"q": "In what year did World War II end?", "fact": "1945", "reference": "World War II ended in 1945."},
    {"q": "What is the boiling point of water in Celsius at sea level?", "fact": "100", "reference": "Water boils at 100 degrees Celsius at sea level."},
    {"q": "What is the currency of Japan?", "fact": "yen", "reference": "The currency of Japan is the yen."},
    {"q": "Who painted the Mona Lisa?", "fact": "da Vinci", "reference": "The Mona Lisa was painted by Leonardo da Vinci."},
]
print(f"Real question set fixed: {len(QUESTIONS)} questions, each with an authoritative fact and reference.")
for item in QUESTIONS:
    print(f"  Q: {item['q']!r} -> fact={item['fact']!r}")

Real question set fixed: 8 questions, each with an authoritative fact and reference.
  Q: 'What is the capital of France?' -> fact='Paris'
  Q: 'What is the chemical symbol for gold?' -> fact='Au'
  Q: 'Who wrote Romeo and Juliet?' -> fact='Shakespeare'
  Q: 'What is the largest planet in our solar system?' -> fact='Jupiter'
  Q: 'In what year did World War II end?' -> fact='1945'
  Q: 'What is the boiling point of water in Celsius at sea level?' -> fact='100'
  Q: 'What is the currency of Japan?' -> fact='yen'
  Q: 'Who painted the Mona Lisa?' -> fact='da Vinci'


## 2. Real Generation: Two Real Answers Per Question, Varied Phrasing

`[REAL]` Two real `gpt-4o-mini` calls per question -- one at low temperature (direct), one at higher temperature with an explicit real instruction to phrase the answer differently -- to produce genuine paraphrase variety, extending Module 01's own 5-paraphrase illustration.

In [3]:
def generate_answer(question, temperature, style_instruction):
    prompt = f"{question} {style_instruction} Keep it to one sentence."
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=60,
    )
    return resp.choices[0].message.content.strip()

generated_items = []
for item in QUESTIONS:
    direct = generate_answer(item["q"], temperature=0.0, style_instruction="Answer directly and simply.")
    varied = generate_answer(item["q"], temperature=0.9, style_instruction="Answer in a full, differently-phrased sentence, varying your wording.")
    generated_items.append({**item, "answer": direct, "variant": "direct"})
    generated_items.append({**item, "answer": varied, "variant": "varied"})
    print(f"Q: {item['q']!r}")
    print(f"  direct: {direct!r}")
    print(f"  varied: {varied!r}")

print(f"\nTotal real generated items: {len(generated_items)}")
print("\n(pending real scoring)")

Q: 'What is the capital of France?'
  direct: 'The capital of France is Paris.'
  varied: 'The capital city of France is Paris.'


Q: 'What is the chemical symbol for gold?'
  direct: 'The chemical symbol for gold is Au.'
  varied: 'The chemical symbol that represents gold is Au.'


Q: 'Who wrote Romeo and Juliet?'
  direct: 'Romeo and Juliet was written by William Shakespeare.'
  varied: 'The playwright William Shakespeare is the author of Romeo and Juliet.'


Q: 'What is the largest planet in our solar system?'
  direct: 'The largest planet in our solar system is Jupiter.'
  varied: 'The most enormous planet in our solar system is Jupiter.'


Q: 'In what year did World War II end?'
  direct: 'World War II ended in 1945.'
  varied: 'The conclusion of World War II took place in the year 1945.'


Q: 'What is the boiling point of water in Celsius at sea level?'
  direct: 'The boiling point of water at sea level is 100 degrees Celsius.'
  varied: 'At sea level, the boiling point of water reaches 100 degrees Celsius.'


Q: 'What is the currency of Japan?'
  direct: 'The currency of Japan is the Japanese yen (JPY).'
  varied: 'The official currency used in Japan is the yen.'


Q: 'Who painted the Mona Lisa?'
  direct: 'The Mona Lisa was painted by Leonardo da Vinci.'
  varied: 'The Mona Lisa was created by the renowned artist Leonardo da Vinci.'

Total real generated items: 16

(pending real scoring)


## 3. Real BLEU-1/ROUGE-1 Scoring and Real Correctness Labeling

`[COMPUTED FROM REAL DATA]` Scoring each real generated answer with Module 02's own BLEU-1 formula against its question's real reference, and applying the pre-stated correctness protocol (Section intro) -- both computed directly from this notebook's own real generated text, not simulated.

In [4]:
def tokenize(s):
    return s.lower().rstrip(".").split()

def bleu1_precision(candidate, reference):
    cand_tokens = tokenize(candidate)
    ref_counts = Counter(tokenize(reference))
    cand_counts = Counter(cand_tokens)
    clipped = sum(min(c, ref_counts[w]) for w, c in cand_counts.items())
    return clipped / len(cand_tokens) if cand_tokens else 0.0

def is_correct(answer, fact):
    """Pre-stated protocol: correct iff the authoritative fact string appears literally
    (case-insensitive) in the real generated answer."""
    return fact.lower() in answer.lower()

for item in generated_items:
    item["bleu1"] = bleu1_precision(item["answer"], item["reference"])
    item["correct"] = is_correct(item["answer"], item["fact"])
    print(f"[{item['variant']:6s}] BLEU-1={item['bleu1']:.3f}, correct={item['correct']}, answer={item['answer']!r}")

n_correct = sum(1 for i in generated_items if i["correct"])
print(f"\nReal correctness rate: {n_correct}/{len(generated_items)} = {n_correct/len(generated_items)*100:.1f}%")
print("\n(pending real interpretation)")

[direct] BLEU-1=1.000, correct=True, answer='The capital of France is Paris.'
[varied] BLEU-1=0.857, correct=True, answer='The capital city of France is Paris.'
[direct] BLEU-1=1.000, correct=True, answer='The chemical symbol for gold is Au.'
[varied] BLEU-1=0.750, correct=True, answer='The chemical symbol that represents gold is Au.'
[direct] BLEU-1=1.000, correct=True, answer='Romeo and Juliet was written by William Shakespeare.'
[varied] BLEU-1=0.455, correct=True, answer='The playwright William Shakespeare is the author of Romeo and Juliet.'
[direct] BLEU-1=1.000, correct=True, answer='The largest planet in our solar system is Jupiter.'
[varied] BLEU-1=0.800, correct=True, answer='The most enormous planet in our solar system is Jupiter.'
[direct] BLEU-1=1.000, correct=True, answer='World War II ended in 1945.'
[varied] BLEU-1=0.417, correct=True, answer='The conclusion of World War II took place in the year 1945.'
[direct] BLEU-1=0.583, correct=True, answer='The boiling point of wa

**Real result:** `gpt-4o-mini` answered all 16 real generated items correctly under the pre-stated protocol — a real `16/16 = 100.0%` correctness rate on this specific easy factual-question set. Real BLEU-1 scores, however, varied substantially: from `0.417` up to `1.000` across the same, uniformly-correct set — purely a function of real phrasing, not correctness, since correctness never varied.

## 4. Real, Exploratory Correlation: BLEU-1 vs. Correctness

`[COMPUTED FROM REAL DATA]` Comparing real mean BLEU-1 scores between the real correct and real incorrect groups -- reported as an exploratory finding given the small real sample size, per the signed-off plan's explicit caveat.

In [5]:
correct_bleu = [i["bleu1"] for i in generated_items if i["correct"]]
incorrect_bleu = [i["bleu1"] for i in generated_items if not i["correct"]]

print(f"Real correct-group BLEU-1: n={len(correct_bleu)}, mean={sum(correct_bleu)/len(correct_bleu):.3f}" if correct_bleu else "No correct items")
print(f"Real incorrect-group BLEU-1: n={len(incorrect_bleu)}, mean={sum(incorrect_bleu)/len(incorrect_bleu):.3f}" if incorrect_bleu else "No incorrect items")

print(f"\nSample size: {len(generated_items)} items -- EXPLORATORY finding only, not a statistically robust result.")
print("\n(pending real interpretation)")

Real correct-group BLEU-1: n=16, mean=0.780
No incorrect items

Sample size: 16 items -- EXPLORATORY finding only, not a statistically robust result.

(pending real interpretation)


## 5. Real Interpretation

`[REAL]` Reported honestly, exactly as it came out: this notebook's pre-stated correctness protocol found **zero incorrect items** among the 16 real generated answers — `gpt-4o-mini` was simply too reliable on this specific easy question set to produce the "wrong but fluent" case Module 02's own hand-worked counterexample demonstrated. This is a real, honest design limitation of *this specific question set*, not a refutation of Module 02's finding — harder, more ambiguous, or more obscure real questions would be needed to reproduce a genuine correctness split at this model's real capability level.

**A real, related finding did emerge, though, directly relevant to the same underlying point.** Splitting by generation style rather than correctness: the `direct` real answers averaged BLEU-1 `0.920`, while the `varied` real answers — genuinely correct, just deliberately reworded — averaged only `0.639`. **Correctness was constant (100%) across both groups**, yet real n-gram overlap dropped by roughly a third purely from real paraphrasing. This is a real, direct, exploratory demonstration of Module 02's central warning — n-gram overlap penalizes real rewording independent of correctness — even though this particular real sample didn't happen to surface a case where a genuinely *wrong* answer outscored a *correct* one. Per the plan's own caveat, this remains an exploratory finding from a small (n=16) real sample, not a statistically robust conclusion — Module 02's own hand-verified 2-example counterexample remains the load-bearing, definitive real evidence.